In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.makedirs('/content/drive/MyDrive/task4_budget', exist_ok=True)
%cd /content/drive/MyDrive/task4_budget
!pwd

/content/drive/MyDrive/task4_budget
/content/drive/MyDrive/task4_budget


In [4]:
import pandas as pd, numpy as np
np.random.seed(3)

categories = ["Marketing","Software","Travel","Office Supplies","Training","Events"]
rows = []
for _ in range(120):
    rows.append([pd.Timestamp("2026-10-01") + pd.Timedelta(days=int(np.random.randint(0, 19))),
                 np.random.choice(categories),
                 round(float(np.random.uniform(50, 1500)), 2)])

exp = pd.DataFrame(rows, columns=["date","category","amount"])
exp.to_csv("expenses.csv", index=False)

totals = exp.groupby("category")["amount"].sum().reindex(categories)
factors = np.array([0.85, 1.20, 0.90, 1.10, 0.80, 1.15])
budgets = pd.DataFrame({"category": categories,
                        "budget": (totals.values * factors).round(-2)})
budgets.to_csv("budgets.csv", index=False)

print("expenses.csv and budgets.csv created")
budgets

import pandas as pd

exp = pd.read_csv("expenses.csv")
bud = pd.read_csv("budgets.csv")

spend = exp.groupby("category")["amount"].sum().reset_index(name="spent")
df = bud.merge(spend, on="category", how="left").fillna({"spent": 0})
df["pct_used"] = (df["spent"] / df["budget"] * 100).round(1)

print("BUDGET vs ACTUAL")
print(df.to_string(index=False), "\n")

over = df[df["spent"] > df["budget"]]

if over.empty:
    alert = "All categories are within budget."
else:
    alert = "BUDGET ALERT: categories over budget\n\n"
    for _, r in over.iterrows():
        alert += (f"- {r['category']}: spent ${r['spent']:,.0f} of ${r['budget']:,.0f} "
                  f"({r['pct_used']}%), over by ${r['spent'] - r['budget']:,.0f}\n")

print(alert)
with open("alert.txt", "w") as f:
    f.write(alert)
print("Saved to alert.txt")

expenses.csv and budgets.csv created
BUDGET vs ACTUAL
       category  budget    spent  pct_used
      Marketing 13800.0 16239.58     117.7
       Software 18600.0 15476.73      83.2
         Travel 13800.0 15345.22     111.2
Office Supplies 14000.0 12746.58      91.0
       Training 14900.0 18638.80     125.1
         Events 19000.0 16529.10      87.0 

BUDGET ALERT: categories over budget

- Marketing: spent $16,240 of $13,800 (117.7%), over by $2,440
- Travel: spent $15,345 of $13,800 (111.2%), over by $1,545
- Training: spent $18,639 of $14,900 (125.1%), over by $3,739

Saved to alert.txt
